# Breast Cancer Classification — Model Training & Testing

**Goal:** Train a binary classifier from the Wisconsin Diagnostic Breast Cancer dataset and save the fitted pipeline as `models/model.joblib`.

> Educational demo only; not for medical diagnosis.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "breast_cancer.csv"
MODEL_PATH = ROOT / "models" / "model.joblib"
print("Dataset:", DATA_PATH)
print("Model output:", MODEL_PATH)

## 2. Load the dataset

The repository includes a reproducible copy of the canonical WDBC dataset also published on Kaggle. The target is encoded as `M` (malignant) or `B` (benign).

In [ ]:
# Prefer the original Kaggle CSV if you place it at data/data.csv.
# Otherwise use the bundled reproducible copy at data/breast_cancer.csv.
kaggle_path = ROOT / "data" / "data.csv"
DATA_PATH = kaggle_path if kaggle_path.exists() else ROOT / "data" / "breast_cancer.csv"
df = pd.read_csv(DATA_PATH)

# Normalize Kaggle's radius_mean-style column names to scikit-learn's feature names.
rename_map = {
    "radius_mean": "mean radius", "texture_mean": "mean texture",
    "perimeter_mean": "mean perimeter", "area_mean": "mean area",
    "smoothness_mean": "mean smoothness", "compactness_mean": "mean compactness",
    "concavity_mean": "mean concavity", "concave points_mean": "mean concave points",
    "symmetry_mean": "mean symmetry", "fractal_dimension_mean": "mean fractal dimension",
    "radius_se": "radius error", "texture_se": "texture error",
    "perimeter_se": "perimeter error", "area_se": "area error",
    "smoothness_se": "smoothness error", "compactness_se": "compactness error",
    "concavity_se": "concavity error", "concave points_se": "concave points error",
    "symmetry_se": "symmetry error", "fractal_dimension_se": "fractal dimension error",
    "radius_worst": "worst radius", "texture_worst": "worst texture",
    "perimeter_worst": "worst perimeter", "area_worst": "worst area",
    "smoothness_worst": "worst smoothness", "compactness_worst": "worst compactness",
    "concavity_worst": "worst concavity", "concave points_worst": "worst concave points",
    "symmetry_worst": "worst symmetry", "fractal_dimension_worst": "worst fractal dimension",
}
df = df.rename(columns=rename_map)
df = df.drop(columns=[c for c in ["id", "Unnamed: 32", "target"] if c in df.columns], errors="ignore")
if "diagnosis" not in df.columns:
    raise ValueError("Dataset must contain a 'diagnosis' column.")
df = df[["diagnosis"] + [c for c in df.columns if c != "diagnosis"]]
print("Loaded:", DATA_PATH)
df.head()

In [ ]:
print("Rows, columns:", df.shape)
print("\\nMissing values:", int(df.isna().sum().sum()))
print("\\nDiagnosis counts:")
print(df["diagnosis"].value_counts())
df.describe().T.head()

## 3. Prepare features and target

- `X`: 30 numeric measurements.
- `y`: `1` = malignant, `0` = benign.
- Keep the test set separate so it is only used for final evaluation.

In [ ]:
feature_names = [c for c in df.columns if c != "diagnosis"]
X = df[feature_names].apply(pd.to_numeric, errors="coerce")
y = (df["diagnosis"] == "M").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Number of features:", len(feature_names))

## 4. Train the model

A scikit-learn `Pipeline` bundles missing-value imputation, scaling, and Logistic Regression into one artifact. This helps ensure inference uses the same preprocessing as training.

In [ ]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
model.fit(X_train, y_train)
print("Training complete.")

## 5. Evaluate on held-out test data

In [ ]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["benign", "malignant"]))
print("Confusion matrix [benign, malignant]:")
print(confusion_matrix(y_test, y_pred))

## 6. Save model and metadata

FastAPI loads this exact file. If you retrain the notebook, restart the API process afterward.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({
    "model": model,
    "feature_names": feature_names,
    "target_labels": {"0": "benign", "1": "malignant"},
}, MODEL_PATH)
print(f"Saved model to: {MODEL_PATH.resolve()}")

## 7. Reload and smoke-test the saved artifact

In [ ]:
loaded = joblib.load(MODEL_PATH)
one_row = X_test.iloc[[0]]
prediction = int(loaded["model"].predict(one_row)[0])
probability = loaded["model"].predict_proba(one_row)[0].max()
print("Predicted class:", "malignant" if prediction == 1 else "benign")
print("Confidence:", round(float(probability), 4))
print("Actual class:", "malignant" if int(y_test.iloc[0]) == 1 else "benign")